In [4]:
import mlflow
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os


In [5]:
import dagshub
dagshub.init(repo_owner="siddharthaganguli0093", repo_name="youtube_Comment_analyzer", mlflow=True)


Accessing as siddharthaganguli0093

Initialized MLflow to track repo "siddharthaganguli0093/youtube_Comment_analyzer"

Repository siddharthaganguli0093/youtube_Comment_analyzer initialized!

In [6]:
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")


2026/07/13 14:14:49 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5e6dbe414fb54c23bd8e365f9aa38662', creation_time=1783932290478, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783932290478, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}, trace_location=None, workspace='default'>

In [7]:
df = pd.read_csv('dataset.csv').dropna(subset=['clean_comment'])
df.shape


(36662, 2)

In [8]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000  # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


2026/07/13 14:16:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2/runs/b5399ff133ba47bd8119901a84bdbe18
🧪 View experiment at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2


2026/07/13 14:19:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2/runs/925ada8342f8410b80976afbbb70fe3c
🧪 View experiment at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2


2026/07/13 14:21:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2/runs/3705385e16554d1197a685a15f4af842
🧪 View experiment at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2


2026/07/13 14:26:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 3)_RandomForest at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2/runs/75824d2cce30470bb44cb16bb1a838a2
🧪 View experiment at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2


2026/07/13 14:29:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 3)_RandomForest at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2/runs/753c6cd9705745e18ef46857cb2a7a18
🧪 View experiment at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/2
